Khóa 7: Cleaning Data in Python

Nhớ lại Khóa 1: bước Transform trong ETL chủ yếu là làm sạch dữ liệu. Ở track SQL, bạn chặn dữ liệu bẩn ngay từ thiết kế bảng (NOT NULL, CHECK, UNIQUE). Nhưng dữ liệu từ file CSV, Excel hay API bên ngoài thì bạn không kiểm soát được, nên phải tự làm sạch bằng pandas. Đây là kỹ năng chiếm phần lớn thời gian của một Data Engineer.

Toàn bộ ví dụ dùng chung bộ dữ liệu bẩn dưới đây, bạn chạy thử được ngay:

In [36]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "id":         [1, 2, 3, 3, 4, 5],
    "name":       [" Nguyen An", "tran binh", "LE CHI", "LE CHI", "Pham Dung", None],
    "salary":     ["25000000", "18000000", "22,000,000", "22,000,000", "abc", "30000000"],
    "department": ["Data", "data", "DATA ", "DATA ", "Sales", "Marketing"],
    "hire_date":  ["2021-03-01", "2022-06-15", "2023-01-10", "2023-01-10", "2020-11-20", "2019-05-05"],
    "age":        [28, 31, 25, 25, 250, 40],
})

Nhìn kỹ sẽ thấy nhiều lỗi: tên viết hoa lộn xộn, lương có dấu phẩy và chữ "abc", phòng ban "Data" / "data" / "DATA " bị coi là 3 giá trị khác nhau, dòng trùng lặp, tuổi 250, và tên bị thiếu.

1. Khám phá dữ liệu trước khi làm sạch

    Không bao giờ sửa dữ liệu khi chưa xem nó. Bốn lệnh đầu tiên luôn nên chạy:

In [37]:
print(df.head())            # xem vài dòng đầu

   id        name      salary department   hire_date  age
0   1   Nguyen An    25000000       Data  2021-03-01   28
1   2   tran binh    18000000       data  2022-06-15   31
2   3      LE CHI  22,000,000      DATA   2023-01-10   25
3   3      LE CHI  22,000,000      DATA   2023-01-10   25
4   4   Pham Dung         abc      Sales  2020-11-20  250


In [38]:
print(df.info())            # kiểu dữ liệu từng cột + số giá trị không null

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          6 non-null      int64 
 1   name        5 non-null      object
 2   salary      6 non-null      object
 3   department  6 non-null      object
 4   hire_date   6 non-null      object
 5   age         6 non-null      int64 
dtypes: int64(2), object(4)
memory usage: 416.0+ bytes
None


In [39]:
print(df.describe())        # thống kê các cột số (min, max, trung bình...)

             id         age
count  6.000000    6.000000
mean   3.000000   66.500000
std    1.414214   90.068307
min    1.000000   25.000000
25%    2.250000   25.750000
50%    3.000000   29.500000
75%    3.750000   37.750000
max    5.000000  250.000000


In [40]:
print(df.isna().sum())      # đếm số giá trị thiếu ở mỗi cột

id            0
name          1
salary        0
department    0
hire_date     0
age           0
dtype: int64


df.info() cho bạn phát hiện ngay lỗi kiểu dữ liệu: cột salary hiện là object (chuỗi) thay vì số. df.describe() cho thấy age có max = 250, vô lý.

Để kiểm tra các cột danh mục:

In [41]:
print(df["department"].value_counts())

department
DATA         2
Data         1
data         1
Sales        1
Marketing    1
Name: count, dtype: int64


Kết quả cho thấy Data, data, DATA được đếm riêng, đây là dấu hiệu dữ liệu không nhất quán. value_counts() giống GROUP BY ... COUNT(*) trong SQL.

2. Lỗi kiểu dữ liệu (Data type problems)

    Cột salary chứa chuỗi nên không tính toán được. Cách xử lý: bỏ ký tự thừa, rồi ép kiểu.

In [42]:
# Bước 1: bỏ dấu phẩy
df["salary"] = df["salary"].str.replace(",", "")

# Bước 2: ép sang số; giá trị không chuyển được ("abc") thành NaN
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")

Tham số errors của to_numeric có 3 lựa chọn:


| Giá trị | Hành vi |
|----------|----------|
| `"raise"` | Báo lỗi và dừng chương trình |
| `"coerce"` | Chuyển thành `NaN` |
| `"ignore"` | Giữ nguyên dữ liệu |

coerce thường được chọn nhất vì pipeline không bị sập, và bạn xử lý các NaN ở bước sau.

Với ngày tháng:

In [43]:
df["hire_date"] = pd.to_datetime(df["hire_date"])

Sau bước này bạn lấy năm, tháng bằng df["hire_date"].dt.year. Nhớ lại Khóa 3 track SQL: bạn từng gặp lỗi so sánh ngày với số nguyên. Chuyển đúng kiểu datetime ngay từ đầu sẽ tránh loại lỗi đó.

Với cột danh mục có ít giá trị khác nhau, dùng category giúp tiết kiệm bộ nhớ:

In [44]:
df["department"] = df["department"].astype("category")

3. Giá trị ngoài phạm vi (Range problems)

    Tuổi 250 là lỗi rõ ràng. Có 3 cách xử lý, tùy ngữ cảnh:

In [45]:
# Cách 1: xóa dòng vô lý
df = df[df["age"] <= 100]

# Cách 2: gán NaN để xử lý sau như dữ liệu thiếu
df.loc[df["age"] > 100, "age"] = np.nan

# Cách 3: ép về giá trị biên (clip)
df["age"] = df["age"].clip(upper=100)

Cách nào đúng phụ thuộc bối cảnh. Với tuổi 250, gán NaN thường hợp lý hơn clip vì ta không biết tuổi thật là bao nhiêu, ép thành 100 cũng là dữ liệu sai. Nhớ lại ràng buộc CHECK (salary > 0) ở Khóa 5 SQL: đây là bản "làm thủ công" của cùng ý tưởng.

4. Dữ liệu trùng lặp (Duplicates)

In [46]:
print(df.duplicated())                       # True/False cho từng dòng
print(df.duplicated().sum())                 # đếm số dòng trùng

df = df.drop_duplicates()                    # xóa dòng trùng hoàn toàn
df = df.drop_duplicates(subset=["id"], keep="first")   # trùng theo cột id, giữ dòng đầu

0    False
1    False
2    False
3     True
5    False
dtype: bool
1


Tham số keep quyết định giữ dòng nào: "first" (mặc định), "last", hoặc False (xóa tất cả các dòng trùng, không giữ dòng nào).

Cẩn thận: dòng trùng "một phần" (cùng id nhưng khác salary) không bị drop_duplicates() mặc định bắt được. Với trường hợp này, thường gộp bằng groupby rồi lấy trung bình hoặc giá trị mới nhất:

In [47]:
df = df.groupby("id", as_index=False).agg({"salary": "max", "name": "first"})

5. Dữ liệu thiếu (Missing data)

    Sau khi to_numeric(errors="coerce"), "abc" đã thành NaN. Ba hướng xử lý:

In [48]:
# Hướng 1: xóa dòng thiếu dữ liệu
df = df.dropna(subset=["name"])

# Hướng 2: điền giá trị cố định
df["name"] = df["name"].fillna("Không rõ")

# Hướng 3: điền bằng thống kê (trung vị hoặc trung bình)
df["salary"] = df["salary"].fillna(df["salary"].median())

Cách chọn dựa vào ý nghĩa của cột:

Cột định danh quan trọng (id, email) bị thiếu: thường xóa dòng, vì dòng đó vô nghĩa.

Cột số có ít giá trị thiếu: điền trung vị (ít bị ảnh hưởng bởi ngoại lai hơn trung bình).

Cột văn bản: điền "Không rõ" để giữ nguyên số dòng.

Luôn ghi lại đã xóa/điền bao nhiêu dòng. Xóa im lặng 30% dữ liệu mà không ai biết là lỗi nghiêm trọng trong pipeline thật.

6. Dữ liệu văn bản và danh mục (Text and categorical)

    Đây là phần quan trọng nhất khóa này. Cột department có "Data", "data", "DATA " là cùng một phòng, cần chuẩn hóa:

In [49]:
df["department"] = df["department"].str.strip().str.lower()
print(df["department"].value_counts())    # giờ chỉ còn: data, sales, marketing

KeyError: 'department'

Các hàm chuỗi hay dùng (đều nằm trong .str):
| Hàm | Tác dụng |
|------|----------|
| `.str.strip()` | Xóa khoảng trắng |
| `.str.lower()` | Chữ thường |
| `.str.upper()` | Chữ hoa |
| `.str.title()` | Viết hoa đầu từ |
| `.str.replace()` | Thay thế |
| `.str.contains()` | Tìm chuỗi |
| `.str.len()` | Độ dài |


Cột name chuẩn hóa như sau:

In [ ]:
df["name"] = df["name"].str.strip().str.title()
# " Nguyen An" -> "Nguyen An", "tran binh" -> "Tran Binh", "LE CHI" -> "Le Chi"

Khi danh mục có nhiều biến thể lộn xộn ("HN", "Ha Noi", "Hanoi"), dùng dict để gộp về một giá trị chuẩn:

In [ ]:
mapping = {"hn": "Hanoi", "ha noi": "Hanoi", "hanoi": "Hanoi"}
df["city"] = df["city"].str.lower().map(mapping)

KeyError: 'city'

Lưu ý: .str. chỉ dùng được trên cột kiểu chuỗi. Nếu cột đã có NaN thì .str vẫn chạy được (NaN được giữ nguyên), nhưng nếu cột là số thì sẽ lỗi.

7. Kiểm tra chéo giữa các cột (Cross-field validation)

    Một số lỗi chỉ lộ ra khi so sánh nhiều cột với nhau. Ví dụ: ngày vào làm không được nằm trong tương lai, hoặc người 20 tuổi không thể có 25 năm kinh nghiệm.

In [ ]:
today = pd.Timestamp.today()
loi = df[df["hire_date"] > today]     # ngày vào làm ở tương lai
print(f"Có {len(loi)} dòng ngày vào làm không hợp lệ")

KeyError: 'hire_date'

Đây là kiểm tra logic nghiệp vụ, giống CHECK liên cột trong SQL.

8. Record linkage: ghép các bản ghi "gần giống nhau"

    Ở Khóa 7 SQL bạn học JOIN, nhưng JOIN yêu cầu khóa khớp chính xác. Thực tế, cùng một công ty có thể được ghi khác nhau ở 2 hệ thống:
```python
ds_a = ["Vinamilk", "FPT Software"]
ds_b = ["Vina Milk", "FPT Softwre", "Viettel"]


JOIN thường sẽ bỏ sót cả hai cặp. Record linkage (hay fuzzy matching) đo mức độ giống nhau giữa các chuỗi rồi ghép các bản ghi gần giống.

Cách đơn giản nhất dùng thư viện có sẵn của Python, không cần cài thêm:

In [ ]:
from difflib import SequenceMatcher

def do_giong(a, b):
    """Trả về số từ 0 đến 1; càng gần 1 càng giống."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print(do_giong("Vinamilk", "Vina Milk"))       # khoảng 0.94
print(do_giong("FPT Software", "FPT Softwre")) # khoảng 0.96
print(do_giong("Vinamilk", "FPT Software"))    # thấp

Quy trình ghép thường như sau:

Với mỗi bản ghi ở bảng A, tính độ giống với các bản ghi ở bảng B.
Chọn bản ghi B có độ giống cao nhất.
Chỉ chấp nhận nếu vượt ngưỡng (ví dụ >= 0.85), không thì coi là không khớp.

In [ ]:
ds_a = ["Vinamilk", "FPT Software"]
ds_b = ["Vina Milk", "FPT Softwre", "Viettel"]

for a in ds_a:
    tot_nhat = max(ds_b, key=lambda b: do_giong(a, b))
    diem = do_giong(a, tot_nhat)
    if diem >= 0.85:
        print(f"{a} <-> {tot_nhat} (giống {diem:.2f})")
    else:
        print(f"{a}: không tìm thấy bản ghi khớp")

Ngưỡng quyết định chất lượng: đặt thấp quá thì ghép nhầm ("Apple" với "Apply"), cao quá thì bỏ sót. Với dữ liệu quan trọng như tài chính, y tế, thường phải có người rà soát các cặp có điểm nằm ở khoảng lưng chừng.

Thư viện chuyên dụng cho việc này là thewfuzz... chính xác là thefuzz (cài bằng pip install thefuzz) và recordlinkage (dùng cho bộ dữ liệu lớn). Nguyên lý giống hệt ví dụ trên.

9. Thứ tự làm sạch điển hình

1 Khi gặp một bộ dữ liệu mới, bạn có thể đi theo thứ tự này:

2 Khám phá: info(), describe(), isna().sum(), value_counts()

3 Sửa kiểu dữ liệu: to_numeric, to_datetime, astype

4 Chuẩn hóa văn bản: strip, lower, map

5 Xử lý ngoài phạm vi: clip, gán NaN, hoặc xóa

6 Xử lý trùng lặp: drop_duplicates

7 Xử lý dữ liệu thiếu: dropna / fillna

8 Kiểm tra lại: chạy lại info() và describe() để chắc chắn kết quả đúng

Bước 7 hay bị bỏ quên nhất. Nhớ bài học ở Project 2 SQL: "chạy không lỗi" chưa có nghĩa là "đúng", nên phải kiểm tra kết quả thực tế.

5. Bài tập thực hành

Dùng bộ dữ liệu sau cho cả 5 bài (bạn chạy thật trong Jupyter):

```python
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [101, 102, 103, 103, 104, 105, 106],
    "customer": [" an nguyen", "BINH TRAN", "chi le", "chi le", "Dung Pham ", None, "em hoang"],
    "amount":   ["500,000", "1200000", "abc", "abc", "750000", "300000", "-50000"],
    "city":     ["HN", "Ha Noi", "hanoi", "hanoi", "HCM", "Hcm", "hn"],
    "order_date": ["2024-01-15", "2024-02-20", "2024-03-05", "2024-03-05", "2024-04-10", "2024-05-01", "2099-12-31"],
})

```
Bài 1 (khám phá): Dùng df.info() và df.isna().sum(), sau đó viết ra bằng lời (dạng comment): có bao nhiêu cột đang sai kiểu dữ liệu, và cột nào có giá trị thiếu?

Bài 2 (kiểu dữ liệu): Làm sạch cột amount: bỏ dấu phẩy, ép sang số bằng to_numeric (errors="coerce"). Chuyển cột order_date sang datetime. In df.dtypes để kiểm tra.

Bài 3 (văn bản và danh mục):

Chuẩn hóa cột customer thành dạng Title Case, không còn khoảng trắng thừa.
Chuẩn hóa cột city để chỉ còn 2 giá trị "Hanoi" và "HCM" (gợi ý: đưa về chữ thường trước, rồi dùng .map() với dict). In value_counts() để kiểm chứng.

Bài 4 (trùng lặp, thiếu, ngoài phạm vi):

Xóa dòng trùng theo order_id.
Xử lý amount âm hoặc NaN: gán tất cả giá trị amount <= 0 thành NaN, sau đó điền các NaN còn lại bằng trung vị của cột.
Xóa dòng có customer bị thiếu.

Bài 5 (kiểm tra chéo và tổng hợp):

Tìm các đơn hàng có order_date lớn hơn ngày hôm nay (gợi ý: pd.Timestamp.today()), in ra số lượng dòng lỗi.
Viết hàm lam_sach_don_hang(df) gộp toàn bộ các bước ở Bài 2, 3, 4 vào một hàm, nhận DataFrame thô và trả về DataFrame đã sạch. Kèm docstring đầy đủ (nhớ Khóa 3).

Làm xong gửi mình chấm như mọi khi. Lần này, hãy gạch chân các động từ trong đề trước khi viết code, rồi tick từng cái sau khi xong. Đó là cách nhanh nhất để không bị thiếu yêu cầu như hai khóa trước.

In [50]:
# Bài 1 (khám phá): Dùng df.info() và df.isna().sum(), sau đó viết ra bằng lời (dạng comment): có bao nhiêu cột đang sai kiểu dữ liệu, và cột nào có giá trị thiếu?

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [101, 102, 103, 103, 104, 105, 106],
    "customer": [" an nguyen", "BINH TRAN", "chi le", "chi le", "Dung Pham ", None, "em hoang"],
    "amount":   ["500,000", "1200000", "abc", "abc", "750000", "300000", "-50000"],
    "city":     ["HN", "Ha Noi", "hanoi", "hanoi", "HCM", "Hcm", "hn"],
    "order_date": ["2024-01-15", "2024-02-20", "2024-03-05", "2024-03-05", "2024-04-10", "2024-05-01", "2099-12-31"],
})

print(df.head())
print(df.info())
print(df.isna().sum())

"""
Co 2 cot sai kieu du lieu la amount va order_date,
cot customer dang thieu gia tri 1 dong
"""

   order_id    customer   amount    city  order_date
0       101   an nguyen  500,000      HN  2024-01-15
1       102   BINH TRAN  1200000  Ha Noi  2024-02-20
2       103      chi le      abc   hanoi  2024-03-05
3       103      chi le      abc   hanoi  2024-03-05
4       104  Dung Pham    750000     HCM  2024-04-10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   order_id    7 non-null      int64 
 1   customer    6 non-null      object
 2   amount      7 non-null      object
 3   city        7 non-null      object
 4   order_date  7 non-null      object
dtypes: int64(1), object(4)
memory usage: 408.0+ bytes
None
order_id      0
customer      1
amount        0
city          0
order_date    0
dtype: int64


'\nCo 2 cot sai kieu du lieu la amount va order_date,\ncot customer dang thieu gia tri 1 dong\n'

In [51]:
# Bài 2 (kiểu dữ liệu): Làm sạch cột amount: bỏ dấu phẩy, ép sang số bằng to_numeric (errors="coerce"). Chuyển cột order_date sang datetime. In df.dtypes để kiểm tra.
df["amount"] = (df["amount"]).str.replace(",","")

df["amount"] = pd.to_numeric(df["amount"],errors="coerce")

df["order_date"] = pd.to_datetime(df["order_date"])

df.dtypes

order_id               int64
customer              object
amount               float64
city                  object
order_date    datetime64[ns]
dtype: object

In [52]:
# Bài 3 (văn bản và danh mục):

# Chuẩn hóa cột customer thành dạng Title Case, không còn khoảng trắng thừa.
# Chuẩn hóa cột city để chỉ còn 2 giá trị "Hanoi" và "HCM" (gợi ý: đưa về chữ thường trước, rồi dùng .map() với dict). In value_counts() để kiểm chứng.

df["customer"]= df["customer"].str.strip().str.title()
mapping = {"hn": "Hanoi", "ha noi": "Hanoi", "hanoi": "Hanoi", "hcm": "HCM"}
df["city"] = df["city"].str.strip().str.lower().map(mapping)
print(df["city"].value_counts(dropna=False))  
print(df["customer"].value_counts())

city
Hanoi    5
HCM      2
Name: count, dtype: int64
customer
Chi Le       2
An Nguyen    1
Binh Tran    1
Dung Pham    1
Em Hoang     1
Name: count, dtype: int64


In [53]:
# Bài 4 (trùng lặp, thiếu, ngoài phạm vi):

# Xóa dòng trùng theo order_id.
# Xử lý amount âm hoặc NaN: gán tất cả giá trị amount <= 0 thành NaN, sau đó điền các NaN còn lại bằng trung vị của cột.
# Xóa dòng có customer bị thiếu.
print(df.duplicated())                       # True/False cho từng dòng
print(df.duplicated().sum())                 # đếm số dòng trùng
df= df.drop_duplicates(subset=["order_id"], keep="first")

df.loc[df["amount"]  <=0, "amount"] = np.nan

df["amount"]= df["amount"].fillna(df["amount"].median())

df= df.dropna(subset=["customer"])

0    False
1    False
2    False
3     True
4    False
5    False
6    False
dtype: bool
1


In [55]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    "order_id": [101, 102, 103, 103, 104, 105, 106],
    "customer": [" an nguyen", "BINH TRAN", "chi le", "chi le", "Dung Pham ", None, "em hoang"],
    "amount":   ["500,000", "1200000", "abc", "abc", "750000", "300000", "-50000"],
    "city":     ["HN", "Ha Noi", "hanoi", "hanoi", "HCM", "Hcm", "hn"],
    "order_date": ["2024-01-15", "2024-02-20", "2024-03-05", "2024-03-05", "2024-04-10", "2024-05-01", "2099-12-31"],
})


In [56]:
# Bài 5 (kiểm tra chéo và tổng hợp):

# Tìm các đơn hàng có order_date lớn hơn ngày hôm nay (gợi ý: pd.Timestamp.today()), in ra số lượng dòng lỗi.
# Viết hàm lam_sach_don_hang(df) gộp toàn bộ các bước ở Bài 2, 3, 4 vào một hàm, nhận DataFrame thô và trả về DataFrame đã sạch. Kèm docstring đầy đủ (nhớ Khóa 3).

def lam_sach_don_hang(df):
    """
    Làm sạch DataFrame đơn hàng thô.

    Các bước: sửa kiểu dữ liệu, chuẩn hóa văn bản, xóa dòng trùng,
    xóa dòng thiếu tên khách hàng, xử lý amount không hợp lệ.

    Params:
        df (pd.DataFrame): dữ liệu thô gồm order_id, customer,
            amount (chuỗi), city, order_date (chuỗi)

    Returns:
        pd.DataFrame: bản đã làm sạch (không sửa df gốc)
    """
    df = df.copy()

    # Kiểu dữ liệu
    df["amount"] = pd.to_numeric(df["amount"].str.replace(",", ""), errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"])

    # Văn bản
    df["customer"] = df["customer"].str.strip().str.title()
    mapping = {"hn": "Hanoi", "ha noi": "Hanoi", "hanoi": "Hanoi", "hcm": "HCM"}
    df["city"] = df["city"].str.strip().str.lower().map(mapping)

    # Trùng lặp và thiếu: loại bỏ trước
    df = df.drop_duplicates(subset=["order_id"], keep="first")
    df = df.dropna(subset=["customer"])

    # Ngoài phạm vi, rồi điền trung vị
    df.loc[df["amount"] <= 0, "amount"] = np.nan
    df["amount"] = df["amount"].fillna(df["amount"].median())

    return df

# Thử với dữ liệu thô (tạo lại DataFrame ban đầu, đặt tên df_raw)
df_sach = lam_sach_don_hang(df_raw)
print(df_sach["city"].value_counts(dropna=False))


city
Hanoi    4
HCM      1
Name: count, dtype: int64
